<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">

# Accelerating Clustering Algorithms to Achieve the Highest Performance #

## Part 4: **H**yper**p**arameter **O**ptimization (HPO) ##

**Table of Contents**
<br>
This notebook shows how to perform hyperparameter optimization for clustering algorithms. This notebook covers the below sections: 

1. [Initial Envrionment Set Up](#Initial-Envrionment-Set-Up)
2. [Data Loading](#Data-Loading)
3. [Import Libraries](#Import-Libraries)
4. [Dask-CUDA Cluster](#Dask-CUDA-Cluster)
5. [K-Means HPO](#K-Means-HPO)
6. [HDBSCAN HPO](#HDBSCAN-HPO)

## Initial Envrionment Set Up

In [1]:
%load_ext cudf.pandas

In [2]:
import pandas as pd
# Sample operation
cudf_df = pd.DataFrame({'a': [1, 2, 3], 'b': [4, 5, 6]})
cudf_df.info()

<class 'cudf.core.dataframe.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   a       3 non-null      int64
 1   b       3 non-null      int64
dtypes: int64(2)
memory usage: 48.0 bytes


In [3]:
import cupy as cp
# Sample operation
guess_ages = cp.zeros((2,3))
type(guess_ages)

cupy.ndarray

## Data Loading

In [4]:
reduced_data_df = pd.read_csv('data/reduced_data_df.csv')

In [5]:
reduced_data_df.shape

(701528, 3)

In [6]:
reduced_data_df = reduced_data_df[:20000]

## Import Libraries

In [7]:
%load_ext cuml.accel

import optuna
import numpy as np
from sklearn.cluster import KMeans
from hdbscan import HDBSCAN
from sklearn.metrics import silhouette_score
from optuna.integration import DaskStorage
from dask.distributed import Client, wait
from dask_cuda import LocalCUDACluster

[2025-08-18 04:16:02.795] [CUML] [info] cuML: Installed accelerator for sklearn.
[2025-08-18 04:16:06.916] [CUML] [info] cuML: Installed accelerator for umap.
[2025-08-18 04:16:06.927] [CUML] [info] cuML: Installed accelerator for hdbscan.
[2025-08-18 04:16:06.927] [CUML] [info] cuML: Successfully initialized accelerator.


## Dask-CUDA Cluster 

In [8]:
# This will use all GPUs on the local host by default
cluster = LocalCUDACluster(threads_per_worker=1, ip="", dashboard_address="8050")
client = Client(cluster)

# Query the client for all connected workers
workers = client.has_what().keys()
n_workers = len(workers)
client

Connection method: Cluster object,Cluster type: dask_cuda.LocalCUDACluster
Dashboard: http://172.18.0.3:8050/status,
Dashboard: http://172.18.0.3:8050/status,Workers: 1
Total threads: 1,Total memory: 15.33 GiB
Status: running,Using processes: True
Comm: tcp://172.18.0.3:38057,Workers: 1
Dashboard: http://172.18.0.3:8050/status,Total threads: 1
Started: Just now,Total memory: 15.33 GiB
Comm: tcp://172.18.0.3:35565,Total threads: 1
Dashboard: http://172.18.0.3:39603/status,Memory: 15.33 GiB
Nanny: tcp://172.18.0.3:36323,


[I 2025-08-18 04:17:00,058] A new study created in memory with name: dask_optuna_kmeans_silhouette_tpe
/opt/conda/lib/python3.10/site-packages/distributed/protocol/pickle.py:90: ExperimentalWarning: DaskStorage is experimental (supported from v3.1.0). The interface can change in the future.
  return pickle.loads(x, buffers=buffers)
[I 2025-08-18 04:23:54,581] A new study created in memory with name: dask_optuna_hdbscan_silhouette_tpe
/opt/conda/lib/python3.10/site-packages/distributed/protocol/pickle.py:90: ExperimentalWarning: DaskStorage is experimental (supported from v3.1.0). The interface can change in the future.
  return pickle.loads(x, buffers=buffers)


**Note**: We're only expecting one GPU worker for our configuration. However, the Dask cluster can be scaled to include multiple GPU workers if needed.

In [9]:
# Get the machine's external IP address
from requests import get

ip=get('https://api.ipify.org').content.decode('utf8')
print(f'Dask dashboard (status) is accessible on http://{ip}:8050')

Dask dashboard (status) is accessible on http://18.204.37.193:8050


## K-Means HPO

In [10]:
# Number of trials for optimization
N_TRIALS = 50

# Convert data to float32
df = reduced_data_df.astype("float32")

# Prepare input features
X = np.array(df)  # Convert to NumPy array for compatibility with scikit-learn metrics

# Study name for Optuna
study_name = "dask_optuna_kmeans_silhouette_tpe"

In [11]:
def train_and_eval(X_param, n_clusters=8, init="k-means++", max_iter=300):
    """
    Splits the given data into train and validation split to train and evaluate the model
    for the given hyperparameters.

    Params
    ______

    X_param:  NumPy array.
              The data to use for training and testing.
    n_clusters: int.
                The number of clusters for KMeans.
    init:      str.
               The initialization method ("k-means++", "random").
    max_iter:  int.
               The maximum number of iterations for KMeans.

    Returns
    score: silhouette score of the fitted model
    """
    kmeans = KMeans(n_clusters=n_clusters, init=init, max_iter=max_iter, n_init = 'auto')
    kmeans.fit(X)
    labels = kmeans.labels_
    score = silhouette_score(X, labels)  # Use silhouette score for evaluation
    return score

In [12]:
# Print initial score with default parameters
print("Score with default parameters : ", train_and_eval(X))

Score with default parameters :  0.98117363


In [13]:
def objective(trial, X_param):
    """
    Objective function for hyperparameter optimization.
    """
    # Suggest hyperparameters
    n_clusters = trial.suggest_int("n_clusters", 2, 20)
    init = trial.suggest_categorical("init", ["k-means++", "random"])
    max_iter = trial.suggest_int("max_iter", 100, 1000)

    # Evaluate with suggested hyperparameters
    score = train_and_eval(X_param, n_clusters=n_clusters, init=init, max_iter=max_iter)
    
    return score  # Maximize silhouette score

In [14]:
# Set up Dask client for distributed computation
storage = DaskStorage()

# Create Optuna study
study = optuna.create_study(
    sampler=optuna.samplers.TPESampler(seed=142),
    study_name=study_name,
    direction="maximize",  # Maximize silhouette score
    storage=storage,
)

# Optimize in parallel on your Dask cluster
n_workers = 4  # Adjust based on your available resources
futures = [
    client.submit(
        study.optimize,
        lambda trial: objective(trial, X),
        n_trials=N_TRIALS // n_workers,
        pure=False,
    )
    for _ in range(n_workers)
]
wait(futures)

# Output results
print(f"Best params: {study.best_params}")
print("Best silhouette score: ", study.best_value)  # Convert back to positive score
print("Number of finished trials: ", len(study.trials))

/tmp/ipykernel_738/550636061.py:2: ExperimentalWarning: DaskStorage is experimental (supported from v3.1.0). The interface can change in the future.
  storage = DaskStorage()
/opt/conda/lib/python3.10/site-packages/distributed/protocol/pickle.py:90: ExperimentalWarning: DaskStorage is experimental (supported from v3.1.0). The interface can change in the future.
  return pickle.loads(x, buffers=buffers)
/opt/conda/lib/python3.10/site-packages/distributed/protocol/pickle.py:90: ExperimentalWarning: DaskStorage is experimental (supported from v3.1.0). The interface can change in the future.
  return pickle.loads(x, buffers=buffers)
[I 2025-08-18 04:17:09,651] Trial 0 finished with value: 0.9706847667694092 and parameters: {'n_clusters': 19, 'init': 'random', 'max_iter': 850}. Best is trial 0 with value: 0.9706847667694092.
[I 2025-08-18 04:17:17,243] Trial 1 finished with value: 0.9876147508621216 and parameters: {'n_clusters': 5, 'init': 'random', 'max_iter': 489}. Best is trial 1 with v

Best params: {'n_clusters': 5, 'init': 'random', 'max_iter': 172}
Best silhouette score:  0.9876982569694519
Number of finished trials:  48


[I 2025-08-18 04:23:03,586] Trial 47 finished with value: 0.986642599105835 and parameters: {'n_clusters': 5, 'init': 'random', 'max_iter': 352}. Best is trial 42 with value: 0.9876982569694519.


## HDBSCAN HPO

In [15]:
# Number of trials for optimization
N_TRIALS = 50

# Convert data to float32
df = reduced_data_df.astype("float32")

# Prepare input features
X = np.array(df)  # Convert to NumPy array for compatibility with scikit-learn metrics

# Study name for Optuna
study_name = "dask_optuna_hdbscan_silhouette_tpe"

In [16]:
def train_and_eval(X_param, min_cluster_size=5, min_samples=10, metric='euclidean'):
    """
    Splits the given data into train and validation split to train and evaluate the model
    for the given hyperparameters.

    Params
    ______

    X_param:  NumPy array.
              The data to use for training and testing.
    n_clusters: int.
                The number of clusters for KMeans.
    init:      str.
               The initialization method ("k-means++", "random").
    max_iter:  int.
               The maximum number of iterations for KMeans.

    Returns
    score: silhouette score of the fitted model
    """
    
    hdbscan = HDBSCAN(min_cluster_size=min_cluster_size, min_samples=min_samples, metric=metric)
    hdbscan.fit(X)
    labels = hdbscan.labels_
    score = silhouette_score(X, labels)  # Use silhouette score for evaluation
    
    return score

In [17]:
# Print initial score with default parameters
print("Score with default parameters : ", train_and_eval(X))

Score with default parameters :  -0.15688638


In [18]:
def objective(trial, X_param):
    """
    Objective function for hyperparameter optimization.
    """
    # Suggest hyperparameters
    min_cluster_size = trial.suggest_int("min_cluster_size", 5, 20)
    min_samples = trial.suggest_int("min_samples", 10, 100, step = 10)
    metric = trial.suggest_categorical("metric", ["euclidean", "l2"])

    # Evaluate with suggested hyperparameters
    score = train_and_eval(X_param, min_cluster_size=min_cluster_size, min_samples=min_samples, metric=metric)
    
    return score  # Maxmize silhouette score

In [19]:
# Set up Dask client for distributed computation
# Create Optuna study
study = optuna.create_study(
    sampler=optuna.samplers.TPESampler(seed=142),
    study_name=study_name,
    direction="maximize",  # Maximize silhouette score
    storage=storage,
)

# Optimize in parallel on your Dask cluster
n_workers = 4  # Adjust based on your available resources
futures = [
    client.submit(
        study.optimize,
        lambda trial: objective(trial, X),
        n_trials=N_TRIALS // n_workers,
        pure=False,
    )
    for _ in range(n_workers)
]
wait(futures)

# Output results
print(f"Best params: {study.best_params}")
print("Best silhouette score: ", study.best_value) 
print("Number of finished trials: ", len(study.trials))

/opt/conda/lib/python3.10/site-packages/distributed/protocol/pickle.py:90: ExperimentalWarning: DaskStorage is experimental (supported from v3.1.0). The interface can change in the future.
  return pickle.loads(x, buffers=buffers)
[I 2025-08-18 04:24:06,180] Trial 0 finished with value: -0.15977421402931213 and parameters: {'min_cluster_size': 19, 'min_samples': 60, 'metric': 'l2'}. Best is trial 0 with value: -0.15977421402931213.
[I 2025-08-18 04:24:14,748] Trial 1 finished with value: -0.19391098618507385 and parameters: {'min_cluster_size': 8, 'min_samples': 20, 'metric': 'euclidean'}. Best is trial 0 with value: -0.15977421402931213.
[I 2025-08-18 04:24:22,702] Trial 2 finished with value: 0.4035184681415558 and parameters: {'min_cluster_size': 11, 'min_samples': 80, 'metric': 'euclidean'}. Best is trial 2 with value: 0.4035184681415558.
[I 2025-08-18 04:24:30,096] Trial 3 finished with value: -0.18367816507816315 and parameters: {'min_cluster_size': 6, 'min_samples': 40, 'metric'

Best params: {'min_cluster_size': 8, 'min_samples': 70, 'metric': 'euclidean'}
Best silhouette score:  0.41500791907310486
Number of finished trials:  48


[I 2025-08-18 04:29:51,839] Trial 47 finished with value: 0.41500791907310486 and parameters: {'min_cluster_size': 15, 'min_samples': 70, 'metric': 'euclidean'}. Best is trial 16 with value: 0.41500791907310486.


**Well Done!** Let's move to the [next notebook](05_BERTopic.ipynb). 

<img src="images/nvidia_header.png" style="margin-left: -30px; width: 300px; float: left;">